In [1]:
# !/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    spherical_grid, spherical_radial_sampling
import equiv_dens.utils.base as utils
from equiv_dens.training.model_loader import load_model

import numpy as np
from functools import partial
import argparse
%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
class LoadFromFile (argparse.Action):
    def __call__ (self, parser, namespace, values, option_string = None):
        with values as f:
            # parse arguments in the file and store them in the target namespace
            parser.parse_args(f.read().split(), namespace)

In [59]:
from equiv_dens.utils.misc import generate_id
from datetime import datetime


# no init coeffs
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae_ccpvtz.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified

model_code = generate_id()
directory = os.path.join(args.save_dir, datetime.utcnow().strftime("%Y-%m-%d_") +
                         model_code)  # generate directory name
# create directories
if not os.path.exists(directory):
    os.makedirs(directory)
# write command line arguments to file (useful for reproducibility)
with open(os.path.join(directory, 'args.txt'), 'w') as f:
    for key in args.__dict__.keys():
        # special case for list input
        if isinstance(args.__dict__[key], list):
            for entry in args.__dict__[key]:
                f.write('--' + key + '=' + str(entry) + "\n")
        else:
            f.write('--' + key + '=' + str(args.__dict__[key]) + "\n")
checkpoint = None
latest_checkpoint = 0
step = 0
args.timing = False
restore = False
data_split_indices = None
# restarts run from latest checkpoint
torch.manual_seed(10)
np.random.seed(10)

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.use_gpu = False
if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

model_ccpvtz = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
best_model_path best_oV82luSd.pth
model code: oV82luSd
args use gpu True
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
level 2
finished init
cg_matrix shape torch.Size([121, 121, 121])
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
orbitals [[(6, 8, 0), (6, 8, 0), (6, 1, 0), (6, 1, 0), (6, 3, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 3)], [(6, 8, 0), (6, 8, 0), (6, 1, 0), (6, 1, 0), (6, 3, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 3)], [(8, 8, 0), (8, 8, 0), (8, 1, 0), (8, 1, 0), (8, 3, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 3)], [(1, 3, 0), (1, 1, 0), (1, 1, 0), (1, 1, 1), (1, 1, 1), (1, 1, 2)], [(1, 3, 0), (1, 1, 0), (

In [60]:
ids = np.random.choice(np.arange(len(dataset)), replace=False, size=(10,))
model_ccpvtz.eval()
with torch.no_grad():
    samples = dataset.get_properties(ids)
    pred_ccpvtz = model_ccpvtz(samples)
    # for j in range(len(pred['spherical_coeffs'])):
    #     print('atom', j)
    #     for orb in pred['spherical_coeffs'][j].keys():
    #         if orb[1] == 0:
    #             print('orb', orb)
    #             print('spherical coeffs', pred['spherical_coeffs'][j][orb])
    print('L0 coeffs pred sum', torch.sum(pred_ccpvtz['L0_coeffs'], 1))
                
                
    print('density integral true', torch.sum(samples['density'] * samples['coord_weights'], 1))
    print('density integral pred ccpvtz', torch.sum(pred_ccpvtz['density'] * pred_ccpvtz['coord_weights'], 1))
    print('true dens shape', samples['density'].shape)
    print('pred dens shape', pred_ccpvtz['density'].shape)
    print('density error pred ccpvtz', torch.sum(torch.abs(pred_ccpvtz['density'] - samples['density']) * pred_ccpvtz['coord_weights'], 1))
    print('max true dens', torch.max(samples['density'][0]))
    print('min true dens', torch.min(samples['density'][0]))
    print('max pred dens', torch.max(pred_ccpvtz['density'][0]))
    print('min pred dens', torch.min(pred_ccpvtz['density'][0]))
    print('max dens diff', torch.max(torch.abs(pred_ccpvtz['density'] - samples['density'])))
    print('min dens diff', torch.min(torch.abs(pred_ccpvtz['density'] - samples['density'])))

properties positions type torch.FloatTensor
L0 coeffs pred sum tensor([ 0.0026,  0.1108,  0.0807, -0.0573,  0.2085,  0.2011,  0.1049,  0.0180,
         0.0024,  0.0730])
density integral true tensor([26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000, 26.0000,
        26.0000, 26.0000])
density integral pred ccpvtz tensor([4309.0034,  121.2491,  157.2443, -174.7930,   75.1503,   75.9509,
         121.3644,  669.3313, 4712.7539,  173.3711])
true dens shape torch.Size([10, 66612])
pred dens shape torch.Size([10, 66612])
density error pred ccpvtz tensor([ 978210.5625,   23351.4453,   32056.9375,   45281.8750,   12395.1436,
          12818.1426,   24665.6094,  143712.8438, 1097925.0000,   35461.6719])
max true dens tensor(278.5768)
min true dens tensor(2.9332e-34)
max pred dens tensor(2.1091e+09)
min pred dens tensor(-12013564.)
max dens diff tensor(2.3621e+09)
min dens diff tensor(6.5728e-06)


In [79]:
from equiv_dens.utils.misc import generate_id
from datetime import datetime


# no init coeffs
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
args.legacy = False
model_code = generate_id()
directory = os.path.join(args.save_dir, datetime.utcnow().strftime("%Y-%m-%d_") +
                         model_code)  # generate directory name
# create directories
if not os.path.exists(directory):
    os.makedirs(directory)
# write command line arguments to file (useful for reproducibility)
with open(os.path.join(directory, 'args.txt'), 'w') as f:
    for key in args.__dict__.keys():
        # special case for list input
        if isinstance(args.__dict__[key], list):
            for entry in args.__dict__[key]:
                f.write('--' + key + '=' + str(entry) + "\n")
        else:
            f.write('--' + key + '=' + str(args.__dict__[key]) + "\n")
checkpoint = None
args.timing = True
latest_checkpoint = 0
step = 0
restore = False
data_split_indices = None
# restarts run from latest checkpoint

torch.manual_seed(10)
np.random.seed(10)

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = True
args.use_gpu = args.use_gpu and torch.cuda.is_available()
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           use_gpu=args.use_gpu,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

old_model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
best_model_path best_izsHHfV1.pth
model code: izsHHfV1
args use gpu True
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
level 2
dataset init grid_spec type torch.cuda.FloatTensor
dataset radial coeffs [[(array([1113.9867719]), array([1.])), (array([369.1623418]), array([1.])), (array([121.79275232]), array([1.])), (array([48.12711454]), array([1.])), (array([20.365074]), array([1.])), (array([8.08835969]), array([1.])), (array([2.50686566]), array([1.])), (array([1.24385374]), array([1.])), (array([0.484499]), array([1.])), (array([0.1918516]), array([1.])), (array([0.07596927]), array([1.])), (array([102.99176249]), array([1.])), (array([28.13259401]), array([1.])), (array([9.83643182]), array([1.])), (array([3.

force props []
no force props ['density']
dtype type <class 'torch.dtype'>
using GPU


In [80]:
ids = np.random.choice(np.arange(len(dataset)), replace=False, size=(10,))
old_model.eval()
with torch.no_grad():
    samples = dataset.get_properties(ids)
    if args.use_gpu:
        for key in samples.keys():
            samples[key] = samples[key].cuda()
    pred_old = old_model(samples)
    # for j in range(len(pred['spherical_coeffs'])):
    #     print('atom', j)
    #     for orb in pred['spherical_coeffs'][j].keys():
    #         if orb[1] == 0:
    #             print('orb', orb)
    #             print('spherical coeffs', pred['spherical_coeffs'][j][orb])
    print('L0 coeffs pred sum', torch.sum(pred_old['L0_coeffs'], 1))
                
                
    print('density integral true', torch.sum(samples['density'] * samples['coord_weights'], 1))
    print('density integral pred old', torch.sum(pred_old['density'] * pred_old['coord_weights'], 1))
    print('true dens shape', samples['density'].shape)
    print('pred dens shape', pred_old['density'].shape)
    print('density error pred old', torch.sum(torch.abs(pred_old['density'] - samples['density']) * pred_old['coord_weights'], 1))
    print('max true dens', torch.max(samples['density'][0]))
    print('min true dens', torch.min(samples['density'][0]))
    print('max pred dens', torch.max(pred_old['density'][0]))
    print('min pred dens', torch.min(pred_old['density'][0]))
    print('max dens diff', torch.max(torch.abs(pred_old['density'] - samples['density'])))
    print('min dens diff', torch.min(torch.abs(pred_old['density'] - samples['density'])))
    
    
print('old model output layer shape', old_model.density_repr_model[1].radial_width[0].weight.shape)

torch.cuda.empty_cache()

properties positions type torch.cuda.FloatTensor
dft network forward start:
Memory allocated 46.0009765625
Memory cached 96.0
repr forward start:
Memory allocated 58.71240234375
Memory cached 96.0
repr forward distances:
Memory allocated 58.7236328125
Memory cached 96.0
repr forward before module blocks:
Memory allocated 59.30859375
Memory cached 96.0
repr forward after module 0 :
Memory allocated 59.3974609375
Memory cached 96.0
repr forward after module 1 :
Memory allocated 59.662109375
Memory cached 122.0
repr forward after module 2 :
Memory allocated 60.1025390625
Memory cached 792.0
sph repr time 1.1166965961456299
repr forward end:
Memory allocated 60.1025390625
Memory cached 792.0
density coeffs forward start:
Memory allocated 59.220703125
Memory cached 792.0
density coeffs forward outputs:
Memory allocated 59.73681640625
Memory cached 792.0
density coeffs forward extract coeffs:
Memory allocated 59.82763671875
Memory cached 792.0
density coeffs time: 0.21857738494873047
density

In [83]:
from equiv_dens.utils.misc import generate_id
from datetime import datetime


# no init coeffs
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_dft8_mae.txt')
print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
args.legacy = True
args.timing = True
model_code = generate_id()
directory = os.path.join(args.save_dir, datetime.utcnow().strftime("%Y-%m-%d_") +
                         model_code)  # generate directory name
# create directories
if not os.path.exists(directory):
    os.makedirs(directory)
# write command line arguments to file (useful for reproducibility)
with open(os.path.join(directory, 'args.txt'), 'w') as f:
    for key in args.__dict__.keys():
        # special case for list input
        if isinstance(args.__dict__[key], list):
            for entry in args.__dict__[key]:
                f.write('--' + key + '=' + str(entry) + "\n")
        else:
            f.write('--' + key + '=' + str(args.__dict__[key]) + "\n")
checkpoint = None
latest_checkpoint = 0
step = 0
restore = False
data_split_indices = None
# restarts run from latest checkpoint

torch.manual_seed(10)
np.random.seed(10)

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = True
args.use_gpu = args.use_gpu and torch.cuda.is_available()
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

if args.cube_grid:
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           L0_coeffs_file=args.L0_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose)

old_model = load_model(args, dataset)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
best_model_path best_cRNEouhL.pth
model code: cRNEouhL
args use gpu True
loading density fromdatasets/ethanol_pyscf_def2svp_dft_f.npy...
loading atoms fromdatasets/ethanol_dft_train.npy...
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'energy', 'forces', 'atom_numbers', 'atom_types'])
level 2
dataset init grid_spec type torch.FloatTensor
dataset radial coeffs [[(array([1113.9867719]), array([1.])), (array([369.1623418]), array([1.])), (array([121.79275232]), array([1.])), (array([48.12711454]), array([1.])), (array([20.365074]), array([1.])), (array([8.08835969]), array([1.])), (array([2.50686566]), array([1.])), (array([1.24385374]), array([1.])), (array([0.484499]), array([1.])), (array([0.1918516]), array([1.])), (array([0.07596927]), array([1.])), (array([102.99176249]), array([1.])), (array([28.13259401]), array([1.])), (array([9.83643182]), array([1.])), (array([3.34905

force props []
no force props ['density']
dtype type <class 'torch.dtype'>
using GPU


In [84]:
ids = np.random.choice(np.arange(len(dataset)), replace=False, size=(10,))
old_model.eval()
with torch.no_grad():
    samples = dataset.get_properties(ids)
    if args.use_gpu:
        for key in samples.keys():
            samples[key] = samples[key].cuda()
    pred_old = old_model(samples)
    # for j in range(len(pred['spherical_coeffs'])):
    #     print('atom', j)
    #     for orb in pred['spherical_coeffs'][j].keys():
    #         if orb[1] == 0:
    #             print('orb', orb)
    #             print('spherical coeffs', pred['spherical_coeffs'][j][orb])
    print('L0 coeffs pred sum', torch.sum(pred_old['L0_coeffs'], 1))
                
                
    print('density integral true', torch.sum(samples['density'] * samples['coord_weights'], 1))
    print('density integral pred old', torch.sum(pred_old['density'] * pred_old['coord_weights'], 1))
    print('true dens shape', samples['density'].shape)
    print('pred dens shape', pred_old['density'].shape)
    print('density error pred old', torch.sum(torch.abs(pred_old['density'] - samples['density']) * pred_old['coord_weights'], 1))
    print('max true dens', torch.max(samples['density'][0]))
    print('min true dens', torch.min(samples['density'][0]))
    print('max pred dens', torch.max(pred_old['density'][0]))
    print('min pred dens', torch.min(pred_old['density'][0]))
    print('max dens diff', torch.max(torch.abs(pred_old['density'] - samples['density'])))
    print('min dens diff', torch.min(torch.abs(pred_old['density'] - samples['density'])))
    
print('old model output layer shape', old_model.density_repr_model[1].radial_width[0].weight.shape)

torch.cuda.empty_cache()

properties positions type torch.FloatTensor
dft network forward start:
Memory allocated 45.57177734375
Memory cached 66.0
repr forward start:
Memory allocated 58.283203125
Memory cached 66.0
repr forward distances:
Memory allocated 58.29443359375
Memory cached 66.0
repr forward before module blocks:
Memory allocated 58.87939453125
Memory cached 66.0
repr forward after module 0 :
Memory allocated 58.96826171875
Memory cached 68.0
repr forward after module 1 :
Memory allocated 59.23291015625
Memory cached 116.0
repr forward after module 2 :
Memory allocated 59.67333984375
Memory cached 786.0
sph repr time 1.1240487098693848
repr forward end:
Memory allocated 59.67333984375
Memory cached 786.0
density coeffs forward start:
Memory allocated 58.79150390625
Memory cached 786.0
density coeffs forward outputs:
Memory allocated 59.3076171875
Memory cached 786.0
density coeffs forward extract coeffs:
Memory allocated 59.3984375
Memory cached 786.0
density coeffs time: 0.19100689888000488
density